In [ ]:
!pip install transformers
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00


# imports

In [ ]:
# for BERT
from transformers import BertTokenizer, BertModel
from bert_score import BERTScorer


In [ ]:
import os
import json
import pandas as pd

In [ ]:
!unzip /content/for_colab_original_model_ques_ans.zip -d /content/original_qwen_ques_ans
!unzip /content/for_colab_ft_model_ques_ans.zip -d /content/ft_qwen_ques_ans

!unzip /content/for_colab_original_model_lis4.zip -d /content/original_model_lis4_ques_ans
!unzip /content/for_colab_ft_model_lis4.zip -d /content/ft_model_lis4_ques_ans

Archive:  /content/for_colab_original_model_ques_ans.zip
replace /content/original_qwen_ques_ans/data/data for uploading in runpod/sentinel/86.32818525312832,23.710052945893406,86.45967786787442,23.817514106826867_2025-04-10_to_2025-04-30/qwen_with_metadata_output.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: Archive:  /content/for_colab_ft_model_ques_ans.zip
replace /content/ft_qwen_ques_ans/data/data for uploading in runpod/landsat/70.05747305980753,22.75584825455804,70.77707755199503,23.347227202657884/70.05747305980753,22.75584825455804,70.77707755199503,23.347227202657884.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
# BERTScore calculation
scorer = BERTScorer(model_type='bert-base-uncased')
# P, R, F1 = scorer.score([candidate], [reference])
# print(f"BERTScore Precision: {P.mean():.4f}, Recall: {R.mean():.4f}, F1: {F1.mean():.4f}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
import json
import re
from typing import Any
import ast

def extract_numeric_answer(model_output: Any):
    """
    Extracts numeric value of `answer` from:
    - ```json { "answer": 1 } ```
    - plain text + JSON block
    - already-parsed dict
    - broken formatting
    Returns int/float or None
    """

    # Case 1: already a dict
    if isinstance(model_output, dict):
        # return model_output.get("answer")
        return model_output

    if not isinstance(model_output, str):
        raise ValueError("model output is not string")

    text = model_output.strip()

    # Case 2: remove code fences ```json ... ```
    text = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE).strip("` \n")

    # Case 3: try strict JSON parsing
    try:
        modified_text = text.replace("null", "None")
        val = ast.literal_eval(modified_text)

        if isinstance(val, dict) and "answer" in val:
            return val

        elif isinstance(val, dict) and "properties" in val:
                if "answer" in val["properties"]:
                    return val["properties"]

    except Exception as e:
        print("ast.literal raising:",e)
        pass


    # Case 4: extract JSON object embedded in text
    json_match = re.search(r"\{[\s\S]*?\}", text)

    if json_match:
        try:
            modified_text = json_match.group().replace("null", "None")

            data = ast.literal_eval(modified_text)

            if isinstance(data, dict) and "answer" in data:
                return data

            elif isinstance(data, dict) and "properties" in data:
                if isinstance(data, dict) and "answer" in data["properties"]:
                    return data["properties"]

        except json.JSONDecodeError:
            pass
        except SyntaxError as e:
            return False

    # Case 5: regex fallback (last resort)
    num_match = re.search(r'"answer"\s*:\s*(-?\d+(?:\.\d+)?)', text)
    if num_match:
        return {"answer": num_match.group(1)}

    return False

def calc_sensor_caption_scores(
    parent_dir_paths_dict
    ):

    in_sensor_correct_record_dict = {
        "without_context":{},
        "with_context":{}
    }

    for sensor,parent_dir_path in parent_dir_paths_dict.items():
        print("="*10,sensor,"="*10)
        in_sensor_correct_record_dict["without_context"].setdefault(sensor, {})
        in_sensor_correct_record_dict["with_context"].setdefault(sensor, {})

        bands_dir_names = os.listdir(parent_dir_path)

        for bands_dir_name in bands_dir_names:
            print("-"*10,bands_dir_name,"-"*10)
            in_sensor_correct_record_dict["without_context"][sensor].setdefault(bands_dir_name, {})
            in_sensor_correct_record_dict["with_context"][sensor].setdefault(bands_dir_name, {})

            grounds_truth_file_path= os.path.join(
                parent_dir_path,
                bands_dir_name,
                "generated_answers_v2.json"
                )

            model_ans_without_context_file_path= os.path.join(
                parent_dir_path,
                bands_dir_name,
                "qwen_output.json"
                )

            model_ans_with_context_file_path= os.path.join(
                parent_dir_path,
                bands_dir_name,
                "qwen_with_metadata_output.json"
                )

            ground_truth_json=dict()
            model_ans_without_context_json=dict()
            model_ans_with_context_json=dict()

            with open(grounds_truth_file_path,"r", encoding="utf-8") as f:
                ground_truth_json=json.load(f)

            with open(model_ans_without_context_file_path,"r", encoding="utf-8") as f:
                model_ans_without_context_json=json.load(f)

            with open(model_ans_with_context_file_path,"r", encoding="utf-8") as f:
                model_ans_with_context_json=json.load(f)

            for ques_cat, ques_dict in ground_truth_json.items():

                if ques_cat in ['image_caption', 'attribute_reasoning']:

                    # print("-"*20,ques_cat,"-"*20)
                    in_sensor_correct_record_dict["without_context"][sensor][bands_dir_name].setdefault(ques_cat,{})
                    in_sensor_correct_record_dict["without_context"][sensor][bands_dir_name][ques_cat]['p']=[]
                    in_sensor_correct_record_dict["without_context"][sensor][bands_dir_name][ques_cat]['r']=[]
                    in_sensor_correct_record_dict["without_context"][sensor][bands_dir_name][ques_cat]['f1']=[]

                    in_sensor_correct_record_dict["with_context"][sensor][bands_dir_name].setdefault(ques_cat,{})
                    in_sensor_correct_record_dict["with_context"][sensor][bands_dir_name][ques_cat]['p']=[]
                    in_sensor_correct_record_dict["with_context"][sensor][bands_dir_name][ques_cat]['r']=[]
                    in_sensor_correct_record_dict["with_context"][sensor][bands_dir_name][ques_cat]['f1']=[]


                    for ques,ground_truth in ques_dict.items():

                        semi_parsed_without_context = extract_numeric_answer(model_ans_without_context_json[ques][0])
                        semi_parsed_with_context = extract_numeric_answer(model_ans_with_context_json[ques][0])

                        fully_parsed_without_context=None
                        fully_parsed_with_context=None

                        if semi_parsed_without_context:
                            fully_parsed_without_context = semi_parsed_without_context['answer']

                        else:
                            answer_removed=model_ans_without_context_json[ques][0].split(""""answer": """)

                            if len(answer_removed)>1:
                                fully_parsed_without_context=model_ans_without_context_json[ques][0].split(""""answer": """)[1]

                            else:
                                fully_parsed_without_context=model_ans_without_context_json[ques][0]

                        if semi_parsed_with_context:
                            fully_parsed_with_context=semi_parsed_with_context['answer']

                        else:
                            answer_removed = model_ans_with_context_json[ques][0].split(""""answer": """)

                            if len(answer_removed)>1:
                                fully_parsed_with_context = model_ans_with_context_json[ques][0].split(""""answer": """)[1]

                            else:
                                fully_parsed_with_context = model_ans_with_context_json[ques][0]


                        without_context_P, without_context_R, without_context_F1 = scorer.score([fully_parsed_without_context], [ground_truth])
                        in_sensor_correct_record_dict["without_context"][sensor][bands_dir_name][ques_cat]['p'].append(without_context_P.tolist()[0])
                        in_sensor_correct_record_dict["without_context"][sensor][bands_dir_name][ques_cat]['r'].append(without_context_R.tolist()[0])
                        in_sensor_correct_record_dict["without_context"][sensor][bands_dir_name][ques_cat]['f1'].append(without_context_F1.tolist()[0])

                        with_context_P, with_context_R, with_context_F1 = scorer.score([fully_parsed_with_context], [ground_truth])
                        in_sensor_correct_record_dict["with_context"][sensor][bands_dir_name][ques_cat]['p'].append(with_context_P.tolist()[0])
                        in_sensor_correct_record_dict["with_context"][sensor][bands_dir_name][ques_cat]['r'].append(with_context_R.tolist()[0])
                        in_sensor_correct_record_dict["with_context"][sensor][bands_dir_name][ques_cat]['f1'].append(with_context_F1.tolist()[0])

    def calc_sensor_scores(in_sensor_data):
        sensor_wise_ques_cat_scores_dict={}
        for sensor, in_bands_dir_data in in_sensor_data.items():

            # question category wise avg P,R,F1 scores for each bands_dir
            temp_dict={}
            sensor_ques_cat_scores_dict={}

            for bands_dir_name, ques_cat_data in in_bands_dir_data.items():

                for ques_cat, ques_cat_meta_data in ques_cat_data.items():

                    temp_dict.setdefault(ques_cat, {})

                    for metric, scores_list in ques_cat_meta_data.items():
                        # print(metric,scores_list)
                        temp_dict[ques_cat].setdefault(metric, [])
                        temp_dict[ques_cat][metric].append(
                            sum(
                                scores_list
                            )/len(scores_list) if scores_list else 0
                        )


            for ques_cat, bands_metric_avg_dict in temp_dict.items():
                # print(ques_cat, bands_metric_avg_dict)
                sensor_ques_cat_scores_dict[ques_cat]={}

                for metric, bands_metric_avg_list in bands_metric_avg_dict.items():


                    sensor_ques_cat_scores_dict[ques_cat][metric]=sum(bands_metric_avg_list)/len(bands_metric_avg_list)



            sensor_wise_ques_cat_scores_dict[sensor]=sensor_ques_cat_scores_dict

        return sensor_wise_ques_cat_scores_dict

    sensor_wise_without_context_scores = calc_sensor_scores(in_sensor_correct_record_dict['without_context'])
    sensor_wise_with_context_scores=calc_sensor_scores(in_sensor_correct_record_dict['with_context'])

    return sensor_wise_without_context_scores, sensor_wise_with_context_scores



# original model scores

In [ ]:
parent_dir_paths_dict = {
    'landsat':"/content/original_qwen_ques_ans/data/data for uploading in runpod/landsat",
    'sentinel':"/content/original_qwen_ques_ans/data/data for uploading in runpod/sentinel",
    'lis3':"/content/original_qwen_ques_ans/data/data for uploading in runpod/lis3",
    'lis4':"/content/original_qwen_ques_ans/data/data for uploading in runpod/lis4"
}

without_context_scores, with_context_scores = calc_sensor_caption_scores(parent_dir_paths_dict=parent_dir_paths_dict)

========== landsat ==========
---------- 78.215,17.263,78.406,17.419 ----------
---------- 70.05747305980753,22.75584825455804,70.77707755199503,23.347227202657884 ----------
---------- 78.68,17.91,78.81,18.02 ----------
---------- 86.32818525312832,23.710052945893406,86.45967786787442,23.817514106826867 ----------
ast.literal raising: invalid syntax (<unknown>, line 1)
ast.literal raising: invalid syntax (<unknown>, line 1)
---------- 77.074928,28.547735,77.095613,28.565602 ----------
ast.literal raising: invalid character '²' (U+00B2) (<unknown>, line 1)
---------- 75.02116881932491,13.604775695171515,75.14869305159489,13.723065478374133 ----------
---------- 74.89050656941924,26.82173124221248,75.23176938680206,27.068408947664345 ----------
---------- 82.49338896860974,22.28268709900715,82.62488158335583,22.385896524592813 ----------
ast.literal raising: invalid syntax (<unknown>, line 1)
---------- 72.03719004155045,26.874580807870533,72.08044870854263,26.909027924511395 ----------

In [ ]:
for sensor_name, sensor_ques_cat_scores in without_context_scores.items():
    print("="*10,sensor_name,"="*10)
    print(sensor_ques_cat_scores)

========== landsat ==========
{'image_caption': {'p': 0.5614103464675801, 'r': 0.5542793859328542, 'f1': 0.5546310919203928}, 'attribute_reasoning': {'p': 0.5719648833785739, 'r': 0.4796260041849954, 'f1': 0.5203535088471004}}
========== sentinel ==========
{'image_caption': {'p': 0.5427021197974682, 'r': 0.5266070647963456, 'f1': 0.5313033876674516}, 'attribute_reasoning': {'p': 0.5631739837782723, 'r': 0.4968538497175489, 'f1': 0.5275487644331796}}
========== lis3 ==========
{'image_caption': {'p': 0.5534580390040691, 'r': 0.552274240897252, 'f1': 0.5467194771537414}, 'attribute_reasoning': {'p': 0.5763797347362225, 'r': 0.49871633373774016, 'f1': 0.5333672761917114}}
========== lis4 ==========
{'image_caption': {'p': 0.5489424479504427, 'r': 0.5490780021581384, 'f1': 0.5434237404002084}, 'attribute_reasoning': {'p': 0.0, 'r': 0.0, 'f1': 0.0}}


In [ ]:
rows = []
for sensor, cat_data in without_context_scores.items():
    row = {"sensor": sensor}
    for ques_cat, metrics in cat_data.items():
        for metric_name, metric_val in metrics.items():
            # key like "image_caption_p", "attribute_reasoning_f1"
            col_name = f"{ques_cat}_{metric_name}"
            row[col_name] = metric_val
    rows.append(row)

# create DataFrame
df = pd.DataFrame(rows)
df = df.set_index("sensor")
df.to_feather("/content/original_without_context_BERT_scores.feather")
# print(df)
df

,image_caption_p,image_caption_r,image_caption_f1,attribute_reasoning_p,attribute_reasoning_r,attribute_reasoning_f1
sensor,,,,,,
landsat,0.561410,0.554279,0.554631,0.571965,0.479626,0.520354
sentinel,0.542702,0.526607,0.531303,0.563174,0.496854,0.527549
lis3,0.553458,0.552274,0.546719,0.576380,0.498716,0.533367
lis4,0.548942,0.549078,0.543424,0.000000,0.000000,0.000000


In [ ]:
for sensor_name, sensor_ques_cat_scores in with_context_scores.items():
    print("="*10,sensor_name,"="*10)
    print(sensor_ques_cat_scores)

========== landsat ==========
{'image_caption': {'p': 0.5880739805953843, 'r': 0.6662544491035598, 'f1': 0.6187155379780701}, 'attribute_reasoning': {'p': 0.7033078372478485, 'r': 0.6860909674848829, 'f1': 0.6934145987033844}}
========== sentinel ==========
{'image_caption': {'p': 0.5498158540576696, 'r': 0.6045600596283164, 'f1': 0.5698539439056601}, 'attribute_reasoning': {'p': 0.6875897135053363, 'r': 0.696945058447974, 'f1': 0.6913058928080967}}
========== lis3 ==========
{'image_caption': {'p': 0.5843637293347945, 'r': 0.6605851552807368, 'f1': 0.6138841899541708}, 'attribute_reasoning': {'p': 0.7026040599896357, 'r': 0.7067303519982558, 'f1': 0.7038566103348365}}
========== lis4 ==========
{'image_caption': {'p': 0.5428396297825707, 'r': 0.5939943169554075, 'f1': 0.5606382261547778}, 'attribute_reasoning': {'p': 0.0, 'r': 0.0, 'f1': 0.0}}


In [ ]:
rows = []
for sensor, cat_data in with_context_scores.items():
    row = {"sensor": sensor}
    for ques_cat, metrics in cat_data.items():
        for metric_name, metric_val in metrics.items():
            # key like "image_caption_p", "attribute_reasoning_f1"
            col_name = f"{ques_cat}_{metric_name}"
            row[col_name] = metric_val
    rows.append(row)

# create DataFrame
df = pd.DataFrame(rows)
df = df.set_index("sensor")
df.to_feather("/content/original_with_context_BERT_scores.feather")
# print(df)
df

,image_caption_p,image_caption_r,image_caption_f1,attribute_reasoning_p,attribute_reasoning_r,attribute_reasoning_f1
sensor,,,,,,
landsat,0.588074,0.666254,0.618716,0.703308,0.686091,0.693415
sentinel,0.549816,0.604560,0.569854,0.687590,0.696945,0.691306
lis3,0.584364,0.660585,0.613884,0.702604,0.706730,0.703857
lis4,0.542840,0.593994,0.560638,0.000000,0.000000,0.000000


# fine tuned model scores

In [ ]:
ft_parent_dir_paths_dict = {
    'landsat':"/content/ft_qwen_ques_ans/data/data for uploading in runpod/landsat",
    'sentinel':"/content/ft_qwen_ques_ans/data/data for uploading in runpod/sentinel",
    'lis3':"/content/ft_qwen_ques_ans/data/data for uploading in runpod/lis3",
    'lis4':"/content/ft_qwen_ques_ans/data/data for uploading in runpod/lis4"
}

ft_without_context_scores, ft_with_context_scores = calc_sensor_caption_scores(parent_dir_paths_dict=ft_parent_dir_paths_dict)

========== landsat ==========
---------- 78.215,17.263,78.406,17.419 ----------
ast.literal raising: invalid syntax (<unknown>, line 1)
ast.literal raising: invalid syntax (<unknown>, line 1)
ast.literal raising: invalid syntax (<unknown>, line 1)
ast.literal raising: invalid syntax (<unknown>, line 1)
---------- 70.05747305980753,22.75584825455804,70.77707755199503,23.347227202657884 ----------
ast.literal raising: invalid syntax (<unknown>, line 1)
ast.literal raising: invalid syntax (<unknown>, line 1)
---------- 78.68,17.91,78.81,18.02 ----------
ast.literal raising: unterminated string literal (detected at line 2) (<unknown>, line 2)
ast.literal raising: invalid syntax (<unknown>, line 1)
ast.literal raising: invalid syntax (<unknown>, line 1)
---------- 86.32818525312832,23.710052945893406,86.45967786787442,23.817514106826867 ----------
ast.literal raising: invalid syntax (<unknown>, line 1)
ast.literal raising: unterminated string literal (detected at line 2) (<unknown>, line 2)

In [ ]:
for sensor_name, sensor_ques_cat_scores in ft_without_context_scores.items():
    print("="*10,sensor_name,"="*10)
    print(sensor_ques_cat_scores)

========== landsat ==========
{'image_caption': {'p': 0.447818764884557, 'r': 0.5484490139143807, 'f1': 0.48934812897018026}, 'attribute_reasoning': {'p': 0.4871897186551775, 'r': 0.4462973581893103, 'f1': 0.46542330724852427}}
========== sentinel ==========
{'image_caption': {'p': 0.4307569656521082, 'r': 0.5167578275182417, 'f1': 0.46711129961269243}, 'attribute_reasoning': {'p': 0.47780886931078775, 'r': 0.46627373354775564, 'f1': 0.4715888031891414}}
========== lis3 ==========
{'image_caption': {'p': 0.44426733226730275, 'r': 0.5349635762663988, 'f1': 0.479620312842039}, 'attribute_reasoning': {'p': 0.4975483990632571, 'r': 0.4582374577338879, 'f1': 0.4763607107675992}}
========== lis4 ==========
{'image_caption': {'p': 0.4446846404009395, 'r': 0.5155391490293874, 'f1': 0.47256870857543415}, 'attribute_reasoning': {'p': 0.0, 'r': 0.0, 'f1': 0.0}}


In [ ]:
rows = []
for sensor, cat_data in ft_without_context_scores.items():
    row = {"sensor": sensor}
    for ques_cat, metrics in cat_data.items():
        for metric_name, metric_val in metrics.items():
            # key like "image_caption_p", "attribute_reasoning_f1"
            col_name = f"{ques_cat}_{metric_name}"
            row[col_name] = metric_val
    rows.append(row)

# create DataFrame
df = pd.DataFrame(rows)
df = df.set_index("sensor")

# print(df)
df.to_feather("/content/fine_tuned_without_context_BERT_scores.feather")
df

,image_caption_p,image_caption_r,image_caption_f1,attribute_reasoning_p,attribute_reasoning_r,attribute_reasoning_f1
sensor,,,,,,
landsat,0.447819,0.548449,0.489348,0.487190,0.446297,0.465423
sentinel,0.430757,0.516758,0.467111,0.477809,0.466274,0.471589
lis3,0.444267,0.534964,0.479620,0.497548,0.458237,0.476361
lis4,0.444685,0.515539,0.472569,0.000000,0.000000,0.000000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
for sensor_name, sensor_ques_cat_scores in ft_with_context_scores.items():
    print("="*10,sensor_name,"="*10)
    print(sensor_ques_cat_scores)

========== landsat ==========
{'image_caption': {'p': 0.5080132729240826, 'r': 0.46756378853959696, 'f1': 0.47217179409095217}, 'attribute_reasoning': {'p': 0.7021321931055614, 'r': 0.5598958390099662, 'f1': 0.620473882981709}}
========== sentinel ==========
{'image_caption': {'p': 0.4690208772995642, 'r': 0.4177418148943356, 'f1': 0.430721898164068}, 'attribute_reasoning': {'p': 0.6213627925940922, 'r': 0.5253959660019193, 'f1': 0.5533001869916916}}
========== lis3 ==========
{'image_caption': {'p': 0.5174019179092004, 'r': 0.4639324967104655, 'f1': 0.4724870639351698}, 'attribute_reasoning': {'p': 0.6190241345992455, 'r': 0.4898112060932013, 'f1': 0.5402862853728808}}
========== lis4 ==========
{'image_caption': {'p': 0.47238796162936425, 'r': 0.4745491238103972, 'f1': 0.4631039723753929}, 'attribute_reasoning': {'p': 0.0, 'r': 0.0, 'f1': 0.0}}


In [ ]:
rows = []
for sensor, cat_data in ft_with_context_scores.items():
    row = {"sensor": sensor}
    for ques_cat, metrics in cat_data.items():
        for metric_name, metric_val in metrics.items():
            # key like "image_caption_p", "attribute_reasoning_f1"
            col_name = f"{ques_cat}_{metric_name}"
            row[col_name] = metric_val
    rows.append(row)

# create DataFrame
df = pd.DataFrame(rows)
df = df.set_index("sensor")
df.to_feather("/content/fine_tuned_with_context_BERT_scores.feather")
df
# print(df)

,image_caption_p,image_caption_r,image_caption_f1,attribute_reasoning_p,attribute_reasoning_r,attribute_reasoning_f1
sensor,,,,,,
landsat,0.508013,0.467564,0.472172,0.702132,0.559896,0.620474
sentinel,0.469021,0.417742,0.430722,0.621363,0.525396,0.553300
lis3,0.517402,0.463932,0.472487,0.619024,0.489811,0.540286
lis4,0.472388,0.474549,0.463104,0.000000,0.000000,0.000000


In [ ]:
pd.read_feather("/content/original_with_context_BERT_scores.feather")

,image_caption_p,image_caption_r,image_caption_f1,attribute_reasoning_p,attribute_reasoning_r,attribute_reasoning_f1
sensor,,,,,,
landsat,0.588074,0.666254,0.618716,0.703308,0.686091,0.693415
sentinel,0.549816,0.604560,0.569854,0.687590,0.696945,0.691306
lis3,0.584364,0.660585,0.613884,0.702604,0.706730,0.703857
lis4,0.542840,0.593994,0.560638,0.000000,0.000000,0.000000
